In [1]:
import pandas as pd
import numpy as np

In [2]:
import optuna
import tensorflow as tf

""" Sequential -  It allows you to create a model by simply adding layers one after another.
This is used for building simple neural networks where the flow of data is straightforward, 
from input to output, through a series of layers."""
from tensorflow.keras.models import Sequential

"""Dense is a fully connected layer, also known as a dense layer, in a neural network.
Each neuron in a dense layer receives input from all neurons in the previous layer."""
from tensorflow.keras.layers import Dense

"""Optimizers computes adaptive learning rates for each parameter"""
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adadelta, Nadam


from sklearn.metrics import roc_auc_score

In [3]:
data = pd.read_csv('churn_modelling.csv')

data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
data.describe(include='all')

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000,10000.000000,10000,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
unique,NaN,NaN,2932,NaN,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,Smith,NaN,France,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,32,NaN,5014,5457,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,5000.50000,1.569094e+07,NaN,650.528800,NaN,NaN,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,NaN,96.653299,NaN,NaN,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,NaN,350.000000,NaN,NaN,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,NaN,584.000000,NaN,NaN,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,NaN,652.000000,NaN,NaN,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,NaN,718.000000,NaN,NaN,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000


In [5]:
data.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [6]:
data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

In [7]:
data = pd.get_dummies(data, drop_first=True)

data

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,1,0,1


In [8]:
targets = data['Exited']

inputs = data.drop(['Exited'],axis=1)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(inputs)

scaled = scaler.transform(inputs)

inputs_scaled = pd.DataFrame(scaled, columns=inputs.columns)

inputs_scaled

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,-0.326221,0.293517,-1.041760,-1.225848,-0.911583,0.646092,0.970243,0.021886,-0.578736,-0.573809,-1.095988
1,-0.440036,0.198164,-1.387538,0.117350,-0.911583,-1.547768,0.970243,0.216534,-0.578736,1.742740,-1.095988
2,-1.536794,0.293517,1.032908,1.333053,2.527057,0.646092,-1.030670,0.240687,-0.578736,-0.573809,-1.095988
3,0.501521,0.007457,-1.387538,-1.225848,0.807737,-1.547768,-1.030670,-0.108918,-0.578736,-0.573809,-1.095988
4,2.063884,0.388871,-1.041760,0.785728,-0.911583,0.646092,0.970243,-0.365276,-0.578736,1.742740,-1.095988
...,...,...,...,...,...,...,...,...,...,...,...
9995,1.246488,0.007457,-0.004426,-1.225848,0.807737,0.646092,-1.030670,-0.066419,-0.578736,-0.573809,0.912419
9996,-1.391939,-0.373958,1.724464,-0.306379,-0.911583,0.646092,0.970243,0.027988,-0.578736,-0.573809,0.912419
9997,0.604988,-0.278604,0.687130,-1.225848,-0.911583,-1.547768,0.970243,-1.008643,-0.578736,-0.573809,-1.095988
9998,1.256835,0.293517,-0.695982,-0.022608,0.807737,0.646092,-1.030670,-0.125231,1.727904,-0.573809,0.912419


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(inputs_scaled, targets, test_size=0.2, random_state=42)

## ANN model

In [13]:
def create_model(trial):
    # Building artificial neural network
    model = Sequential()

     # we add 2 hidden layers and 1 output layer
    model.add(Dense(units=trial.suggest_int('units_layer1', 6, 32), activation='relu'))
    model.add(Dense(units=trial.suggest_int('units_layer2', 6, 32), activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))

    # Suggest hyperparameters for the optimizer
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop', 'adagrad'])
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    
    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        optimizer = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    elif optimizer_name == 'adagrad':
        optimizer = Adagrad(learning_rate=learning_rate)
    
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['AUC'])
    
    return model

## Optimize hyperparameters with Optuna

In [14]:
def optimal(trial):
    
    # Suggest the number of epochs and batch size
    epochs = trial.suggest_int('epochs', 10, 50)
    batch_size = trial.suggest_int('batch_size', 16, 64)
    
    model = create_model(trial)
    
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size)
    
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)
    
    return auc

study = optuna.create_study(direction='maximize')
study.optimize(optimal, n_trials=10)

print(f"Best trial: {study.best_trial.value}")
print(f"Best hyperparameters: {study.best_trial.params}")

[I 2024-12-16 22:56:49,191] A new study created in memory with name: no-name-383c2817-4b8e-44a5-8f91-61d6ed64bfa3
/var/folders/br/yvkmbr3121n8fd9q6wdmyqfm0000gp/T/ipykernel_30415/3644950783.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


Epoch 1/48
146/146 [==============================] - 0s 390us/step - loss: 0.6547 - auc: 0.4662
Epoch 2/48
146/146 [==============================] - 0s 325us/step - loss: 0.6041 - auc: 0.4772
Epoch 3/48
146/146 [==============================] - 0s 328us/step - loss: 0.5763 - auc: 0.4950
Epoch 4/48
146/146 [==============================] - 0s 335us/step - loss: 0.5576 - auc: 0.5150
Epoch 5/48
146/146 [==============================] - 0s 335us/step - loss: 0.5440 - auc: 0.5313
Epoch 6/48
146/146 [==============================] - 0s 329us/step - loss: 0.5333 - auc: 0.5492
Epoch 7/48
146/146 [==============================] - 0s 324us/step - loss: 0.5246 - auc: 0.5663
Epoch 8/48
146/146 [==============================] - 0s 331us/step - loss: 0.5172 - auc: 0.5808
Epoch 9/48
146/146 [==============================] - 0s 330us/step - loss: 0.5108 - auc: 0.5957
Epoch 10/48
146/146 [==============================] - 0s 329us/step - loss: 0.5051 - auc: 0.6092
Epoch 11/48
146/146 [========

[I 2024-12-16 22:56:52,047] Trial 0 finished with value: 0.785681599744122 and parameters: {'epochs': 48, 'batch_size': 55, 'units_layer1': 29, 'units_layer2': 15, 'optimizer': 'adagrad', 'learning_rate': 0.0012843568508620919}. Best is trial 0 with value: 0.785681599744122.
/var/folders/br/yvkmbr3121n8fd9q6wdmyqfm0000gp/T/ipykernel_30415/3644950783.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


Epoch 1/26
308/308 [==============================] - 0s 378us/step - loss: 0.9973 - auc: 0.4906
Epoch 2/26
308/308 [==============================] - 0s 302us/step - loss: 0.9646 - auc: 0.4888
Epoch 3/26
308/308 [==============================] - 0s 299us/step - loss: 0.9433 - auc: 0.4872
Epoch 4/26
308/308 [==============================] - 0s 300us/step - loss: 0.9266 - auc: 0.4854
Epoch 5/26
308/308 [==============================] - 0s 294us/step - loss: 0.9126 - auc: 0.4843
Epoch 6/26
308/308 [==============================] - 0s 288us/step - loss: 0.9004 - auc: 0.4829
Epoch 7/26
308/308 [==============================] - 0s 302us/step - loss: 0.8895 - auc: 0.4819
Epoch 8/26
308/308 [==============================] - 0s 289us/step - loss: 0.8797 - auc: 0.4811
Epoch 9/26
308/308 [==============================] - 0s 331us/step - loss: 0.8707 - auc: 0.4806
Epoch 10/26
308/308 [==============================] - 0s 340us/step - loss: 0.8624 - auc: 0.4801
Epoch 11/26
308/308 [========

[I 2024-12-16 22:56:54,919] Trial 1 finished with value: 0.48624259956836424 and parameters: {'epochs': 26, 'batch_size': 26, 'units_layer1': 15, 'units_layer2': 21, 'optimizer': 'adagrad', 'learning_rate': 0.0001832527149256269}. Best is trial 0 with value: 0.785681599744122.
/var/folders/br/yvkmbr3121n8fd9q6wdmyqfm0000gp/T/ipykernel_30415/3644950783.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


...

Epoch 1/10
286/286 [==============================] - 0s 367us/step - loss: 0.4042 - auc: 0.7915
Epoch 2/10
286/286 [==============================] - 0s 325us/step - loss: 0.3561 - auc: 0.8462
Epoch 3/10
286/286 [==============================] - 0s 325us/step - loss: 0.3460 - auc: 0.8555
Epoch 4/10
286/286 [==============================] - 0s 322us/step - loss: 0.3422 - auc: 0.8595
Epoch 5/10
286/286 [==============================] - 0s 331us/step - loss: 0.3366 - auc: 0.8644
Epoch 6/10
286/286 [==============================] - 0s 327us/step - loss: 0.3361 - auc: 0.8657
Epoch 7/10
286/286 [==============================] - 0s 326us/step - loss: 0.3348 - auc: 0.8662
Epoch 8/10
286/286 [==============================] - 0s 321us/step - loss: 0.3332 - auc: 0.8675
Epoch 9/10
286/286 [==============================] - 0s 327us/step - loss: 0.3304 - auc: 0.8691
Epoch 10/10
63/63 [==============================] - 0s 270us/step


[I 2024-12-16 22:57:07,945] Trial 8 finished with value: 0.8559467089752056 and parameters: {'epochs': 10, 'batch_size': 28, 'units_layer1': 21, 'units_layer2': 29, 'optimizer': 'adam', 'learning_rate': 0.00500404372800007}. Best is trial 2 with value: 0.8592971905673493.
/var/folders/br/yvkmbr3121n8fd9q6wdmyqfm0000gp/T/ipykernel_30415/3644950783.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


Epoch 1/23
174/174 [==============================] - 0s 384us/step - loss: 0.4339 - auc: 0.7575
Epoch 2/23
174/174 [==============================] - 0s 313us/step - loss: 0.3627 - auc: 0.8403
Epoch 3/23
174/174 [==============================] - 0s 318us/step - loss: 0.3497 - auc: 0.8514
Epoch 4/23
174/174 [==============================] - 0s 330us/step - loss: 0.3438 - auc: 0.8578
Epoch 5/23
174/174 [==============================] - 0s 322us/step - loss: 0.3410 - auc: 0.8609
Epoch 6/23
174/174 [==============================] - 0s 320us/step - loss: 0.3389 - auc: 0.8629
Epoch 7/23
174/174 [==============================] - 0s 321us/step - loss: 0.3363 - auc: 0.8653
Epoch 8/23
174/174 [==============================] - 0s 335us/step - loss: 0.3352 - auc: 0.8653
Epoch 9/23
174/174 [==============================] - 0s 324us/step - loss: 0.3323 - auc: 0.8694
Epoch 10/23
174/174 [==============================] - 0s 320us/step - loss: 0.3306 - auc: 0.8701
Epoch 11/23
174/174 [========

[I 2024-12-16 22:57:09,505] Trial 9 finished with value: 0.8620000601693292 and parameters: {'epochs': 23, 'batch_size': 46, 'units_layer1': 23, 'units_layer2': 22, 'optimizer': 'rmsprop', 'learning_rate': 0.004002208728285055}. Best is trial 9 with value: 0.8620000601693292.


Best trial: 0.8620000601693292
Best hyperparameters: {'epochs': 23, 'batch_size': 46, 'units_layer1': 23, 'units_layer2': 22, 'optimizer': 'rmsprop', 'learning_rate': 0.004002208728285055}


In [15]:
best_params = study.best_trial.params

best_params

{'epochs': 23,
 'batch_size': 46,
 'units_layer1': 23,
 'units_layer2': 22,
 'optimizer': 'rmsprop',
 'learning_rate': 0.004002208728285055}

In [16]:
# Train the final model with the best hyperparameters

best_model = Sequential()
best_model.add(Dense(units=best_params['units_layer1'], activation='relu'))
best_model.add(Dense(units=best_params['units_layer2'], activation='relu'))
best_model.add(Dense(1, activation='sigmoid'))

In [17]:
if best_params['optimizer'] == 'adam':
    best_optimizer = Adam(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'sgd':
    best_optimizer = SGD(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'rmsprop':
    best_optimizer = RMSprop(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'adagrad':
    best_optimizer = Adagrad(learning_rate=best_params['learning_rate'])


In [18]:
best_model.compile(optimizer=best_optimizer, loss='binary_crossentropy', metrics=['AUC'])

In [19]:
def evaluate(model, X_train, y_train, X_test, y_test):
    
    model.fit(X_train, y_train, epochs=23, batch_size=best_params['batch_size'])
    
    '''Predictions and probabilities for the training set'''
    
    y_train_prob = model.predict(X_train)

    '''Predictions and probabilities for the test set'''
    
    y_test_prob = model.predict(X_test)

    '''Calculate metrics for the training set''' 
    
    roc_train_prob = roc_auc_score(y_train, y_train_prob)
    gini_train_prob = roc_train_prob * 2 - 1
    

    '''Calculate metrics for the test set'''
    
    roc_test_prob = roc_auc_score(y_test, y_test_prob)
    gini_test_prob = roc_test_prob * 2 - 1
    

    results = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'Gini': [gini_train_prob * 100, gini_test_prob * 100],
    
    })

    return results

In [20]:
evaluate(best_model, X_train, y_train, X_test, y_test)

Epoch 1/20
174/174 [==============================] - 0s 390us/step - loss: 0.4345 - auc: 0.7555
Epoch 2/20
174/174 [==============================] - 0s 343us/step - loss: 0.3718 - auc: 0.8310
Epoch 3/20
174/174 [==============================] - 0s 365us/step - loss: 0.3506 - auc: 0.8507
Epoch 4/20
174/174 [==============================] - 0s 342us/step - loss: 0.3456 - auc: 0.8561
Epoch 5/20
174/174 [==============================] - 0s 352us/step - loss: 0.3416 - auc: 0.8598
Epoch 6/20
174/174 [==============================] - 0s 359us/step - loss: 0.3365 - auc: 0.8649
Epoch 7/20
174/174 [==============================] - 0s 358us/step - loss: 0.3339 - auc: 0.8667
Epoch 8/20
174/174 [==============================] - 0s 348us/step - loss: 0.3317 - auc: 0.8690
Epoch 9/20
174/174 [==============================] - 0s 347us/step - loss: 0.3278 - auc: 0.8714
Epoch 10/20
174/174 [==============================] - 0s 340us/step - loss: 0.3270 - auc: 0.8725
Epoch 11/20
174/174 [========

,Dataset,Gini
0,Train,79.417689
1,Test,70.045491


## deploy

In [45]:
data = {
    'CreditScore': [600, 750, 680, 720, 590],
    'Age': [40, 35, 50, 29, 45],
    'Tenure': [3, 7, 5, 2, 10],
    'Balance': [60000, 85000, 0, 50000, 120000],
    'NumOfProducts': [2, 1, 2, 3, 1],
    'HasCrCard': [1, 0, 1, 1, 0],
    'IsActiveMember': [1, 0, 1, 1, 0],
    'EstimatedSalary': [50000, 70000, 40000, 80000, 30000],
    'Geography_Germany': [0, 1, 0, 0, 0],
    'Geography_Spain': [0, 0, 1, 0, 0],
    'Gender_Male': [1, 0, 1, 1, 0]
}

# Create a DataFrame
cust = pd.DataFrame(data)

cust

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,600,40,3,60000,2,1,1,50000,0,0,1
1,750,35,7,85000,1,0,0,70000,1,0,0
2,680,50,5,0,2,1,1,40000,0,1,1
3,720,29,2,50000,3,1,1,80000,0,0,1
4,590,45,10,120000,1,0,0,30000,0,0,0


In [46]:
scaler.fit(cust)

scaled = scaler.transform(cust)

df_test_scaled = pd.DataFrame(scaled, columns=cust.columns)

df_test_scaled

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,-1.0677,0.0272,-0.8361,-0.0756,0.2673,0.8165,0.8165,-0.2157,-0.5000,-0.5000,0.8165
1,1.2876,-0.6522,0.5574,0.5542,-1.0690,-1.2247,-1.2247,0.8627,2.0000,-0.5000,-1.2247
2,0.1884,1.3860,-0.1393,-1.5869,0.2673,0.8165,0.8165,-0.7548,-0.5000,2.0000,0.8165
3,0.8165,-1.4675,-1.1844,-0.3275,1.6036,0.8165,0.8165,1.4018,-0.5000,-0.5000,0.8165
4,-1.2247,0.7066,1.6025,1.4358,-1.0690,-1.2247,-1.2247,-1.2940,-0.5000,-0.5000,-1.2247


In [47]:
cust['predicted_churn'] = best_model.predict(df_test_scaled)


cust

1/1 [==============================] - 0s 11ms/step


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male,predicted_churn
0,600,40,3,60000,2,1,1,50000,0,0,1,0.2265
1,750,35,7,85000,1,0,0,70000,1,0,0,0.4541
2,680,50,5,0,2,1,1,40000,0,1,1,0.0374
3,720,29,2,50000,3,1,1,80000,0,0,1,0.0199
4,590,45,10,120000,1,0,0,30000,0,0,0,0.2252
